# Pareto Front Filtering and Solution Combination Analysis

This notebook implements advanced Pareto optimality filtering for multi-objective railroad construction scheduling solutions, focusing on efficient solution space reduction while maintaining solution quality diversity.

## Analysis Framework:

### 1. **Multi-Objective Optimization Context**
- **7 Optimization Objectives**: Driver violations, commute distance, transport machines, transport attachments, machine count, worker count, attachment count
- **Solution Space Reduction**: Eliminates dominated solutions while preserving Pareto-optimal trade-offs
- **Computational Efficiency**: Reduces solution set size for practical decision-making

### 2. **Two-Stage Pareto Filtering Strategy**
- **Stage 1**: 2D Pareto filtering on attachment-related objectives (Transport Attachments, Attachments)
- **Stage 2**: Full multi-dimensional Pareto filtering across all 7 objectives
- **Solution Combination**: Creates hybrid solutions by combining optimal components

### 3. **Hybrid Solution Generation**
- **Decomposed Optimization**: Separates worker/machine planning from attachment allocation
- **Cross-Product Strategy**: Combines Pareto-optimal attachment solutions with base solutions
- **Composite Solution IDs**: Creates new solution identifiers reflecting component origins

### 4. **Data Processing Pipeline**
- **Pareto Dominance Detection**: Identifies non-dominated solutions in multi-objective space
- **Solution Expansion**: Generates all viable combinations of optimization components
- **Final Filtering**: Applies comprehensive Pareto filtering to combined solution set

## Real-Life Problem Context:
- **Instance**: July 1, 2024 railroad construction scenario
- **Scale**: 400+ construction operations with complex resource constraints
- **Optimization Goals**: Minimize costs, violations, and resource usage while maintaining feasibility
- **Practical Application**: Enables decision-makers to select from a reduced set of high-quality, non-dominated solutions

The filtering process significantly reduces solution complexity while maintaining optimization quality, supporting efficient decision-making in practical railroad construction scheduling scenarios.

In [ ]:
"""
Multi-Objective Pareto Front Filtering and Solution Combination

This cell implements a sophisticated two-stage Pareto filtering process for railroad
construction scheduling solutions, combining partial optimization results to create
comprehensive solution alternatives while maintaining Pareto optimality.
"""

import pandas as pd
import numpy as np

# === Data Loading and Initialization ===
# Load complete Pareto front data containing all optimization objectives
# File contains solutions with multiple objectives for construction scheduling
df = pd.read_csv("ParetoFront.csv")  # Adjust path if necessary

# === Multi-Objective Definition ===
# Define all optimization objectives for comprehensive analysis
# Each objective represents a different aspect of construction scheduling optimization
all_objectives = [
    "Driver Violation",      # Constraint violations related to driver scheduling rules
    "Commute Distance",      # Total worker commuting costs and travel time
    "Transport Machines",    # Machine transportation costs between construction sites
    "Transport Attachments", # Equipment/attachment transportation costs
    "Machines",             # Total machine resource requirements
    "Workers",              # Human resource allocation and count
    "Attachments"           # Equipment/attachment resource allocation count
]

# === Stage 1: 2D Pareto Front Extraction for Attachment Optimization ===
def pareto_front_2d(points):
    """
    Compute 2D Pareto front for bi-objective optimization problems.
    
    Identifies non-dominated solutions in 2-dimensional objective space using
    efficient dominance checking. A solution dominates another if it is better
    or equal in all objectives and strictly better in at least one objective.
    
    Args:
        points: Array of 2D points representing objective values
        
    Returns:
        Array of Pareto-optimal points (non-dominated solutions)
    """
    points = np.array(points)  # Convert to numpy array for efficient operations
    is_efficient = np.ones(points.shape[0], dtype=bool)  # Initialize all as efficient
    
    # Iterate through each point to check dominance relationships
    for i, c in enumerate(points):
        if is_efficient[i]:  # Only check if point is still considered efficient
            # Check dominance: point c dominates others if c <= others in all dimensions
            # and c < others in at least one dimension
            is_efficient[is_efficient] = (
                np.any(points[is_efficient] < c, axis=1)  # Strictly better in at least one objective
                | np.all(points[is_efficient] == c, axis=1)  # Or identical solutions
            )
            is_efficient[i] = True  # Current point remains efficient
    
    return points[is_efficient]  # Return only Pareto-optimal points

# Apply 2D Pareto filtering to attachment-related objectives
# Focus on transport and allocation trade-offs for equipment optimization
pareto_points = pareto_front_2d(df[["Transport Attachments", "Attachments"]].values)

# Create DataFrame with Pareto-optimal attachment combinations
df_pareto_attach = pd.DataFrame(
    pareto_points, columns=["Transport Attachments", "Attachments"]
).drop_duplicates()  # Remove any duplicate solutions

# === Solution ID Mapping for Pareto-Optimal Attachment Combinations ===
# Identify original solution IDs corresponding to Pareto-optimal attachment combinations
# This mapping enables reconstruction of complete solutions from components
pareto_attach_ids = df.merge(
    df_pareto_attach, 
    on=["Transport Attachments", "Attachments"]
)[["Solution ID", "Transport Attachments", "Attachments"]]

# Display Pareto-optimal attachment combinations with their solution IDs
print("\n🆔 Solution IDs der Pareto-optimalen (Transport Attachments, Attachments):")
print(pareto_attach_ids.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

# === Stage 2: Solution Space Expansion through Cross-Product Combination ===
# Create expanded solution set by combining base solutions with Pareto-optimal attachments
# This generates all possible combinations of worker/machine plans with optimal equipment allocation

# Remove attachment columns from base solutions for cross-product combination
df_base = df.drop(columns=["Transport Attachments", "Attachments"])

# Perform cross-product merge: every base solution combined with every Pareto attachment solution
df_expanded = df_base.merge(pareto_attach_ids, how="cross")

# === Composite Solution ID Generation ===
# Create new solution identifiers reflecting component origins
# Format: {base_solution_id}_{attachment_solution_id}
df_expanded["Solution ID"] = (
    df_expanded["Solution ID_x"].astype(str) + "_" + 
    df_expanded["Solution ID_y"].astype(str)
)

# === Data Structure Reorganization ===
# Reorder columns with new composite Solution ID at the beginning
# Remove intermediate merge columns created during cross-product operation
cols = ["Solution ID"] + [
    col for col in df_expanded.columns 
    if col not in ["Solution ID", "Solution ID_x", "Solution ID_y", "Solution ID_X_Y"]
]
df_expanded = df_expanded[cols]

# === Stage 3: Comprehensive Multi-Dimensional Pareto Filtering ===
def pareto_filter_nd(df, objective_cols):
    """
    Apply n-dimensional Pareto filtering to remove dominated solutions.
    
    Performs comprehensive dominance checking across all specified objectives,
    retaining only solutions that are not dominated by any other solution
    in the multi-objective space.
    
    Args:
        df: DataFrame containing solutions and objective values
        objective_cols: List of column names representing optimization objectives
        
    Returns:
        DataFrame containing only non-dominated (Pareto-optimal) solutions
    """
    values = df[objective_cols].values  # Extract objective values for dominance checking
    is_efficient = np.ones(values.shape[0], dtype=bool)  # Initialize all as efficient
    
    # Comprehensive dominance checking across all objectives
    for i, v in enumerate(values):
        if is_efficient[i]:  # Only check if solution is still considered efficient
            # Multi-dimensional dominance check
            is_efficient[is_efficient] = (
                np.any(values[is_efficient] < v, axis=1)  # Better in at least one objective
                | np.all(values[is_efficient] == v, axis=1)  # Or identical in all objectives
            )
            is_efficient[i] = True  # Current solution remains efficient
    
    return df[is_efficient].reset_index(drop=True)  # Return filtered DataFrame

# Apply comprehensive Pareto filtering to expanded solution set
df_final_pareto = pareto_filter_nd(df_expanded, all_objectives)

# === Final Data Organization ===
# Sort columns in logical order for analysis and presentation
sort_cols = [
    "Solution ID",          # Composite identifier
    "Orders",              # Problem size metrics
    "Order Items",         # Problem complexity metrics
    "Driver Violation",    # Constraint satisfaction objectives
    "Commute Distance",    # Cost objectives
    "Transport Machines",  # Transportation objectives
    "Transport Attachments",
    "Machines",           # Resource allocation objectives
    "Workers",
    "Attachments"
]
df_final_pareto = df_final_pareto[sort_cols]

# === Analysis Insights and Results Summary ===
print("-"*40)
print("🔍 Insights zur Paretoanalyse\n" + "-"*40)
print(f"📦 Ursprüngliche Lösungen:         {len(df)}")                    # Original solution count
print(f"🎯 Pareto-Kombinationen (2D):      {len(df_pareto_attach)}")     # 2D Pareto-optimal combinations
print(f"🧩 Erweiterte Lösungskombis:       {len(df_expanded)}")          # Total expanded combinations
print(f"✅ Nicht-dominierte Endlösungen:   {len(df_final_pareto)}")      # Final Pareto-optimal solutions
print(f"❌ Entfernte (dominierte) Lösungen: {len(df_expanded) - len(df_final_pareto)}\n")  # Dominated solutions removed

# Display 2D Pareto-optimal attachment combinations
print("📊 Pareto-Kombinationen (Anbaugeräte):")
print(df_pareto_attach.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

# === Output Generation ===
# Save filtered Pareto front for further analysis and decision-making
df_final_pareto.to_csv("ParetoFront_filtered.csv", index=False)
print(f"\n💾 Datei gespeichert unter: ParetoFront_filtered.csv")

# === Process Benefits ===
# This filtering process provides:
# 1. Significant reduction in solution space complexity
# 2. Maintained solution quality through Pareto optimality preservation
# 3. Hybrid solution generation combining optimal components
# 4. Comprehensive multi-objective trade-off analysis


🆔 Solution IDs der Pareto-optimalen (Transport Attachments, Attachments):
 Solution ID  Transport Attachments  Attachments
         210                1462.17           57
         178                4932.71           44
----------------------------------------
🔍 Insights zur Paretoanalyse
----------------------------------------
📦 Ursprüngliche Lösungen:         979
🎯 Pareto-Kombinationen (2D):      2
🧩 Erweiterte Lösungskombis:       1958
✅ Nicht-dominierte Endlösungen:   38
❌ Entfernte (dominierte) Lösungen: 1920

📊 Pareto-Kombinationen (Anbaugeräte):
 Transport Attachments  Attachments
               1462.17         57.0
               4932.71         44.0

💾 Datei gespeichert unter: ParetoFront_filtered.csv


In [ ]:
"""
Filtered Pareto Front Visualization and Analysis

This cell provides comprehensive display and analysis of the final filtered
Pareto front, showing the complete set of non-dominated solutions after
multi-objective optimization and solution combination processes.
"""

# === Complete Solution Set Display ===
# Display the complete filtered Pareto front with all solution details
# This provides a comprehensive view of remaining non-dominated solutions
print("\n📄 Inhalt der gefilterten Paretofront:")
print("-" * 40)

# Show all filtered solutions with complete objective value information
# Each row represents a unique non-dominated solution in the multi-objective space
print(df_final_pareto.to_string(index=False))

print("-" * 40)
print("Lösungen:", len(df_final_pareto))  # Display total count of remaining solutions

# === Analysis Benefits ===
# This visualization provides:
# 1. Complete transparency of final solution set
# 2. Easy comparison of objective trade-offs between solutions
# 3. Clear identification of solution diversity in multi-objective space
# 4. Support for informed decision-making in practical deployment scenarios


📄 Inhalt der gefilterten Paretofront:
----------------------------------------
Solution ID  Orders  Order Items  Driver Violation  Commute Distance  Transport Machines  Transport Attachments  Machines  Workers  Attachments
      1_178      85          884               365         323458.00            64794.07                4932.71        65      118           44
      1_210      85          884               365         323458.00            64794.07                1462.17        65      118           57
      2_178      85          884               366         322743.89            64829.81                4932.71        65      118           44
      2_210      85          884               366         322743.89            64829.81                1462.17        65      118           57
      3_178      85          884               367         322760.73            64814.37                4932.71        65      118           44
      3_210      85          884               367      

In [ ]:
"""
Complete Solution Reconstruction and Export

This cell reconstructs complete optimization solutions by combining component parts
from the filtered Pareto front, creating deployable solution files that contain
all necessary routing and scheduling information for practical implementation.
"""

import json
import pandas as pd
from pathlib import Path

# === File Path Configuration ===
# Define input and output paths for solution reconstruction process
solutions_path = Path("pareto_solutions.json")      # Original complete solutions repository
filtered_path = Path("ParetoFront_filtered.csv")   # Filtered Pareto front identifiers
output_path = Path("pareto_solutions_filtered.json")  # Output for reconstructed solutions

# === Data Loading Phase ===
# Load source data for solution reconstruction process

# 1. Load complete solution repository containing all routing and scheduling details
# This JSON file contains detailed worker routes, machine plans, and attachment schedules
with open(solutions_path, "r", encoding="utf-8") as f:
    solutions = json.load(f)

# 2. Load filtered Pareto front containing composite solution identifiers
# CSV contains solution IDs in format "base_id_attachment_id" representing optimal combinations
df_filtered = pd.read_csv(filtered_path)

# === Data Validation ===
# Ensure required data structure exists for reconstruction process
if "Solution ID" not in df_filtered.columns:
    raise ValueError("Die CSV-Datei muss eine Spalte 'Solution ID' enthalten.")

# === Solution Reconstruction Process ===
# Reconstruct complete solutions by combining optimal components
combined = {}  # Dictionary to store reconstructed solutions

# Process each filtered Pareto-optimal solution
for _, row in df_filtered.iterrows():
    combined_id = row["Solution ID"]          # Composite identifier (e.g., "1_178")
    base_id, attach_id = combined_id.split("_")  # Split into component identifiers
    
    # === Component Validation ===
    # Verify that both component solutions exist in the source repository
    if base_id not in solutions or attach_id not in solutions:
        print(f"⚠️ ID {combined_id} enthält ungültige Referenz: {base_id} oder {attach_id} nicht gefunden.")
        continue
    
    # === Component Extraction ===
    # Extract optimal components from source solutions
    base = solutions[base_id]      # Base solution (worker and machine routing)
    attach = solutions[attach_id]  # Attachment solution (equipment allocation)
    
    # === Solution Reconstruction ===
    # Combine components to create complete hybrid solution
    # This maintains optimization quality while creating new solution combinations
    combined[combined_id] = {
        # Primary routing components from base solution
        "worker_route_plan": base["worker_route_plan"],      # Worker scheduling and routing
        "machine_route_plan": base["machine_route_plan"],    # Machine allocation and routing
        
        # Optimal attachment allocation from specialized solution
        "attachment_route_plan": attach["attachment_route_plan"],  # Equipment assignment
        
        # Metadata tracking component origins for traceability
        "combined_from": {
            "worker_machine_id": base_id,    # Source of worker/machine optimization
            "attachment_id": attach_id       # Source of attachment optimization
        }
    }

# === Output Generation ===
# Save reconstructed solutions in deployable JSON format
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

# Provide reconstruction summary
print(f"✅ {len(combined)} kombinierte Lösungen gespeichert unter: {output_path}")

# === Reconstruction Benefits ===
# This process provides:
# 1. Complete deployable solutions from Pareto-optimal components
# 2. Hybrid optimization combining best aspects of different solutions
# 3. Traceability of component origins for analysis and validation
# 4. Reduced solution space while maintaining optimization quality
# 5. JSON format suitable for integration with scheduling systems

✅ 38 kombinierte Lösungen gespeichert unter: pareto_solutions_filtered.json
